In [5]:
import os

In [6]:
%pwd

'd:\\Siam\\End-to-End-ML-Project-with-MLflow\\research'

In [7]:
os.chdir("../")

In [8]:
%pwd

'd:\\Siam\\End-to-End-ML-Project-with-MLflow'

In [9]:
import pandas as pd

In [10]:
df = pd.read_csv("artifacts/data_ingestion/winequality-red.csv")
df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1599 entries, 0 to 1598
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1599 non-null   float64
 1   volatile acidity      1599 non-null   float64
 2   citric acid           1599 non-null   float64
 3   residual sugar        1599 non-null   float64
 4   chlorides             1599 non-null   float64
 5   free sulfur dioxide   1599 non-null   float64
 6   total sulfur dioxide  1599 non-null   float64
 7   density               1599 non-null   float64
 8   pH                    1599 non-null   float64
 9   sulphates             1599 non-null   float64
 10  alcohol               1599 non-null   float64
 11  quality               1599 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 150.0 KB


In [12]:
df.isnull().sum()

fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
dtype: int64

In [13]:
df.shape

(1599, 12)

In [14]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataValidationConfig:
    root_dir: Path
    STATUS_FILE: str
    unzip_data_dir: Path
    all_schema: dict

In [15]:
from mlProject.constants import *
from mlProject.utils.common import read_yaml, create_directories

In [28]:
class ConfigurationManager:
    def __init__(
            self,
            config_filepath = CONFIG_FILE_PATH,
            params_filepath = PARAMS_FILE_PATH,
            schema_filepath = SCHEMA_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories(self.config["artifacts_root"])
    
    def get_data_validation_config(self) -> DataValidationConfig:
        config = self.config["data_validation"]
        schema = self.schema["COLUMNS"]

        create_directories([config["root_dir"]])

        data_validation_config = DataValidationConfig(
            root_dir=config["root_dir"],
            STATUS_FILE=config["status_file"],
            unzip_data_dir=config["unzip_data_dir"],
            all_schema=schema
        )
        return data_validation_config

In [17]:
import os
from mlProject.logger import logging
from mlProject.exception import CustomException
import sys

In [36]:
class DataValidation:
    def __init__(self, config: DataValidationConfig):
        self.config = config

    def validate_all_columns(self) -> bool:
        try:
            data = pd.read_csv(self.config.unzip_data_dir)

            # actual columns in data
            data_cols = list(data.columns)

            expected_schema = self.config.all_schema

            validation_status = True

            # 1. Check if all expected columns exist
            for col in expected_schema.keys():
                if col not in data_cols:
                    validation_status = False
                    break

            # 2. Check if extra columns exist in data
            if validation_status:
                for col in data_cols:
                    if col not in expected_schema:
                        validation_status = False
                        break

            # 3. Check datatype match
            if validation_status:
                for col, expected_dtype in expected_schema.items():
                    actual_dtype = str(data[col].dtype)

                    if actual_dtype != str(expected_dtype):
                        validation_status = False
                        break

            # Write status once
            with open(self.config.STATUS_FILE, "w") as f:
                f.write(f"Validation status: {validation_status}")

            return validation_status

        except Exception as e:
            raise CustomException(e, sys)

In [37]:
try:
    config = ConfigurationManager()
    data_validation_config = config.get_data_validation_config()
    data_validation = DataValidation(config=data_validation_config)
    data_validation.validate_all_columns()
except Exception as e:
    raise CustomException(e, sys)

[2026-04-09 15:32:52,149: INFO: common: yaml file: D:\Siam\End-to-End-ML-Project-with-MLflow\config\config.yaml loaded successfully]
[2026-04-09 15:32:52,150: INFO: common: yaml file: D:\Siam\End-to-End-ML-Project-with-MLflow\params.yaml loaded successfully]
[2026-04-09 15:32:52,153: INFO: common: yaml file: D:\Siam\End-to-End-ML-Project-with-MLflow\schema.yaml loaded successfully]
[2026-04-09 15:32:52,154: INFO: common: created directory at: a]
[2026-04-09 15:32:52,155: INFO: common: created directory at: r]
[2026-04-09 15:32:52,156: INFO: common: created directory at: t]
[2026-04-09 15:32:52,157: INFO: common: created directory at: i]
[2026-04-09 15:32:52,158: INFO: common: created directory at: f]
[2026-04-09 15:32:52,159: INFO: common: created directory at: a]
[2026-04-09 15:32:52,161: INFO: common: created directory at: c]
[2026-04-09 15:32:52,162: INFO: common: created directory at: t]
[2026-04-09 15:32:52,163: INFO: common: created directory at: s]
[2026-04-09 15:32:52,163: INFO